### 1.3.7.7. Forward-Mode Automatic Differentiation

$$
f(a + b\,\varepsilon) = f(a) + f'(a)\,b\,\varepsilon, \qquad \varepsilon^{2} = 0 .
$$

**Explanation:**

Forward-mode AD evaluates a function and its derivative together by carrying a *dual number* $a + b\varepsilon$, where $\varepsilon$ is a formal symbol with $\varepsilon^2 = 0$. Each elementary operation propagates the value in the real part and the derivative in the $\varepsilon$-part, applying the [chain rule](../02_Single_Variable_Differentiation/03_chain_rule.ipynb) automatically — seeding $b = 1$ for the input returns $f'$ in one forward pass. Unlike a finite difference it is exact (no step-size error), and unlike a symbolic derivative it never builds a formula. It is the [JVP](./06_jacobian_vector_and_vector_jacobian_products.ipynb) primitive and is efficient when the number of inputs is small.

**Properties:**
- Arithmetic on dual numbers encodes the differentiation rules: $(a+b\varepsilon)(c+d\varepsilon) = ac + (ad+bc)\varepsilon$ is the product rule.
- Cost is comparable to one function evaluation per input direction.

**Numerical Example:**

Differentiate $f(x) = x^2 + \sin x$ at $x = 2$ using a dual number with seed $1$, i.e. $x = 2 + 1\varepsilon$:

$$
x^2 = (2 + \varepsilon)^2 = 4 + 4\varepsilon, \qquad \sin x = \sin 2 + \cos 2\,\varepsilon,
$$

$$
f(x) = (4 + \sin 2) + (4 + \cos 2)\,\varepsilon .
$$

The real part $4 + \sin 2 \approx 4.909$ is $f(2)$ and the $\varepsilon$-part $4 + \cos 2 \approx 3.584$ is $f'(2)$, matching $f'(x) = 2x + \cos x$ at $x = 2$.

In [1]:
import math

class Dual:
    def __init__(self, value, derivative=0.0):
        self.value = value
        self.derivative = derivative

    def __add__(self, other):
        other = other if isinstance(other, Dual) else Dual(other)
        return Dual(self.value + other.value, self.derivative + other.derivative)

    def __mul__(self, other):
        other = other if isinstance(other, Dual) else Dual(other)
        return Dual(self.value * other.value,
                    self.derivative * other.value + self.value * other.derivative)

    def __pow__(self, power):
        return Dual(self.value ** power,
                    power * self.value ** (power - 1) * self.derivative)

def dual_sin(number):
    return Dual(math.sin(number.value), math.cos(number.value) * number.derivative)

input_dual = Dual(2.0, 1.0)
output_dual = input_dual ** 2 + dual_sin(input_dual)

print("f(2)  (real part)    =", output_dual.value)
print("f'(2) (epsilon part) =", output_dual.derivative)
print("check 2x + cos x at 2 =", 2 * 2 + math.cos(2))

f(2)  (real part)    = 4.909297426825682
f'(2) (epsilon part) = 3.5838531634528574
check 2x + cos x at 2 = 3.5838531634528574


**Equivalent CasADi implementation:**

CasADi performs the same forward-mode sweep with `ca.jtimes` (a JVP with seed $1$), giving $f'(2)$ directly.

In [2]:
import casadi as ca

x = ca.SX.sym("x")
f = x**2 + ca.sin(x)
forward_derivative = ca.jtimes(f, x, ca.DM(1))
derivative_function = ca.Function("forward", [x], [forward_derivative])

print("f'(2) via forward-mode AD =", float(derivative_function(2.0)))

f'(2) via forward-mode AD = 3.5838531634528574


**References:**

[📘 Griewank, A., & Walther, A. (2008). *Evaluating Derivatives* (2nd ed.). SIAM.](https://epubs.siam.org/doi/book/10.1137/1.9780898717761)  
[📗 Baydin, A. G., Pearlmutter, B. A., Radul, A. A., & Siskind, J. M. (2018). *Automatic Differentiation in Machine Learning: a Survey*. JMLR 18(153).](https://jmlr.org/papers/v18/17-468.html)

---

[⬅️ Previous: Jacobian-Vector and Vector-Jacobian Products](./06_jacobian_vector_and_vector_jacobian_products.ipynb) | [Next: Reverse-Mode Automatic Differentiation ➡️](./08_reverse_mode_automatic_differentiation.ipynb)